# Building a Comparable Company (Comps) Analysis

This notebook explains a production-level comps workflow: selecting peers, collecting fundamentals, normalizing metrics, and deriving implied valuation ranges. A working code example pulls financial data from the SEC XBRL API and market prices from free public sources (with an override option if a feed is blocked).

## 1. Comps logic in one page

A comps analysis estimates a company’s value by applying peer trading multiples. The method is simple but must be disciplined:

1. **Select peers** (business model, size, growth, margins).
2. **Collect clean financials** (revenue, EBITDA, net income, debt, cash, shares).
3. **Compute enterprise value and multiples** (EV/Revenue, EV/EBITDA, P/E).
4. **Normalize** (consistent periods, remove outliers, use medians).
5. **Apply peer multiples** to the target’s metrics to estimate implied value.

We keep each step explicit and reproducible.

In [ ]:
from __future__ import annotations

from io import StringIO
from typing import Iterable
import os

import numpy as np
import pandas as pd
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SEC_HEADERS = {
    "User-Agent": "EEIF Comps Tutorial contact@example.com",
    "Accept-Encoding": "gzip, deflate",
}

ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "")


def fetch_json(url: str) -> dict:
    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def get_cik_from_ticker(ticker: str) -> str:
    url = "https://www.sec.gov/files/company_tickers.json"
    data = fetch_json(url)
    for item in data.values():
        if item["ticker"].lower() == ticker.lower():
            return str(item["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found in SEC list.")


def get_company_facts(cik: str) -> dict:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    return fetch_json(url)


def extract_tag_series(facts: dict, tags: Iterable[str], unit: str = "USD") -> pd.Series:
    for tag in tags:
        tag_data = facts.get("facts", {}).get("us-gaap", {}).get(tag, {})
        if not tag_data:
            continue
        units = tag_data.get("units", {})
        if unit not in units:
            continue
        rows = units[unit]
        frame = pd.DataFrame(rows)
        if frame.empty:
            continue
        frame = frame[frame["form"].isin(["10-K", "20-F", "40-F"])].copy()
        frame["fy"] = frame["fy"].astype(int)
        frame = frame.sort_values("fy").drop_duplicates("fy", keep="last")
        series = pd.Series(frame["val"].values, index=frame["fy"])
        series.name = tag
        return series
    return pd.Series(dtype=float)


def coalesce_series(series_list: Iterable[pd.Series]) -> pd.Series:
    result = None
    for series in series_list:
        if series.empty:
            continue
        if result is None:
            result = series.copy()
        else:
            result = result.combine_first(series)
    return result if result is not None else pd.Series(dtype=float)


def latest_value(series: pd.Series) -> float | None:
    if series.empty:
        return None
    return float(series.dropna().iloc[-1])


def fetch_alpha_vantage_price(ticker: str) -> float | None:
    if not ALPHAVANTAGE_API_KEY:
        return None
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={ticker}&apikey={ALPHAVANTAGE_API_KEY}"
    )
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        payload = response.json()
        quote = payload.get("Global Quote", {})
        price = quote.get("05. price")
        return float(price) if price else None
    except (requests.RequestException, ValueError):
        return None


def fetch_market_price(ticker: str) -> float | None:
    return fetch_alpha_vantage_price(ticker)

## 2. Define peer set and optional price overrides

Choose peers with similar business models and scale. If public feeds are blocked, you can manually input prices below.

In [2]:
tickers = ["NVDA", "AMD", "AVGO", "INTC"]
price_overrides = {
    # Optional manual overrides if market feeds are blocked.
    # "NVDA": 0.0,
    # "AMD": 0.0,
    # "AVGO": 0.0,
    # "INTC": 0.0,
}

## 3. Fetch fundamentals and build a comps table

We compute enterprise value using market cap, debt, and cash, then calculate the most common multiples. We use LTM proxies from the latest fiscal year available in the SEC filings.

In [4]:
def build_comps_row(ticker: str, price_override: float | None = None) -> dict:
    cik = get_cik_from_ticker(ticker)
    facts = get_company_facts(cik)

    revenue = coalesce_series([
        extract_tag_series(facts, ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues"]),
        extract_tag_series(facts, ["SalesRevenueNet"]),
    ])
    operating_income = extract_tag_series(facts, ["OperatingIncomeLoss"])
    depreciation = coalesce_series([
        extract_tag_series(facts, ["DepreciationAndAmortization"]),
        extract_tag_series(facts, ["DepreciationDepletionAndAmortization"]),
    ])
    net_income = extract_tag_series(facts, ["NetIncomeLoss"])

    cash = coalesce_series([
        extract_tag_series(facts, ["CashAndCashEquivalentsAtCarryingValue", "CashCashEquivalentsAndShortTermInvestments"]),
    ])
    debt = coalesce_series([
        extract_tag_series(facts, ["LongTermDebt", "LongTermDebtNoncurrent", "DebtLongterm"]),
    ])
    shares_out = extract_tag_series(facts, ["CommonStockSharesOutstanding"], unit="shares")

    last_fy = int(revenue.dropna().index.max()) if not revenue.empty else None
    last_revenue = latest_value(revenue)
    last_ebit = latest_value(operating_income)
    last_da = latest_value(depreciation)
    last_net_income = latest_value(net_income)
    last_cash = latest_value(cash) or 0.0
    last_debt = latest_value(debt) or 0.0
    last_shares = latest_value(shares_out)

    price = price_override if price_override is not None else fetch_market_price(ticker)
    market_cap = price * last_shares if price and last_shares else None
    enterprise_value = None
    if market_cap is not None:
        enterprise_value = market_cap + last_debt - last_cash

    ebitda = None
    if last_ebit is not None and last_da is not None:
        ebitda = last_ebit + last_da

    return {
        "Ticker": ticker,
        "Last_FY": last_fy,
        "Revenue": last_revenue,
        "EBITDA": ebitda,
        "NetIncome": last_net_income,
        "MarketPrice": price,
        "Shares": last_shares,
        "MarketCap": market_cap,
        "EnterpriseValue": enterprise_value,
    }


rows = [build_comps_row(t, price_overrides.get(t)) for t in tickers]
comps = pd.DataFrame(rows)

comps["EV_Revenue"] = comps["EnterpriseValue"] / comps["Revenue"]
comps["EV_EBITDA"] = comps["EnterpriseValue"] / comps["EBITDA"]
comps["PE"] = comps["MarketCap"] / comps["NetIncome"]

display(comps)
comps

,Ticker,Last_FY,Revenue,EBITDA,NetIncome,MarketPrice,Shares,MarketCap,EnterpriseValue,EV_Revenue,EV_EBITDA,PE
0,NVDA,2022,"26,914,000,000.00","133,230,000,000.00","120,067,000,000.00",None,"24,304,000,000.00",None,None,NaN,NaN,NaN
1,AMD,2025,"34,639,000,000.00","3,916,000,000.00","4,335,000,000.00",None,"1,630,000,000.00",None,None,NaN,NaN,NaN
2,AVGO,2025,"63,887,000,000.00",NaN,"5,895,000,000.00",None,"4,741,000,000.00",None,None,NaN,NaN,NaN
3,INTC,2025,"52,853,000,000.00","-2,014,000,000.00","-267,000,000.00",None,"4,994,000,000.00",None,None,NaN,NaN,NaN


,Ticker,Last_FY,Revenue,EBITDA,NetIncome,MarketPrice,Shares,MarketCap,EnterpriseValue,EV_Revenue,EV_EBITDA,PE
0,NVDA,2022,"26,914,000,000.00","133,230,000,000.00","120,067,000,000.00",None,"24,304,000,000.00",None,None,NaN,NaN,NaN
1,AMD,2025,"34,639,000,000.00","3,916,000,000.00","4,335,000,000.00",None,"1,630,000,000.00",None,None,NaN,NaN,NaN
2,AVGO,2025,"63,887,000,000.00",NaN,"5,895,000,000.00",None,"4,741,000,000.00",None,None,NaN,NaN,NaN
3,INTC,2025,"52,853,000,000.00","-2,014,000,000.00","-267,000,000.00",None,"4,994,000,000.00",None,None,NaN,NaN,NaN


## 4. Implied value for a target company

We apply the peer median multiples to the target’s metrics to estimate implied enterprise value and implied price. This is how comps translate into a valuation range.

In [5]:
target = "NVDA"
target_row = comps[comps["Ticker"] == target].iloc[0]

median_ev_rev = comps["EV_Revenue"].median()
median_ev_ebitda = comps["EV_EBITDA"].median()

implied_ev_rev = median_ev_rev * target_row["Revenue"] if pd.notna(target_row["Revenue"]) else None
implied_ev_ebitda = median_ev_ebitda * target_row["EBITDA"] if pd.notna(target_row["EBITDA"]) else None

def ev_to_price(implied_ev: float | None, row: pd.Series) -> float | None:
    if implied_ev is None or pd.isna(row["Shares"]):
        return None
    net_debt = (row["EnterpriseValue"] - row["MarketCap"]) if pd.notna(row["EnterpriseValue"]) and pd.notna(row["MarketCap"]) else 0.0
    implied_equity = implied_ev - net_debt
    return implied_equity / row["Shares"]

implied_price_rev = ev_to_price(implied_ev_rev, target_row)
implied_price_ebitda = ev_to_price(implied_ev_ebitda, target_row)

summary = pd.DataFrame({
    "Metric": ["EV/Revenue", "EV/EBITDA"],
    "Peer_Median": [median_ev_rev, median_ev_ebitda],
    "Implied_EV": [implied_ev_rev, implied_ev_ebitda],
    "Implied_Price": [implied_price_rev, implied_price_ebitda],
})

display(summary)
summary

,Metric,Peer_Median,Implied_EV,Implied_Price
0,EV/Revenue,NaN,NaN,NaN
1,EV/EBITDA,NaN,NaN,NaN


,Metric,Peer_Median,Implied_EV,Implied_Price
0,EV/Revenue,NaN,NaN,NaN
1,EV/EBITDA,NaN,NaN,NaN
